In [32]:
# 1. CONFIGURAÇÃO E ESTADO GLOBAL

import sympy
from sympy.logic.boolalg import Implies, And, Or, Not
from sympy import Symbol
from sympy import parse_expr
from IPython.display import display, HTML
import re
import sys
import pandas as pd

# Aumenta o limite de recursão para o motor de busca (DN)
sys.setrecursionlimit(3000)

# Variável de cache global para otimizar o motor DN (Memoization)
TARGET_CACHE = {}


# 2. UTILITÁRIOS LÓGICOS (LOGIC UTILS)

def parse_formula_logica(formula_str):
    """Parseia a string da fórmula para um objeto SymPy."""
    if not formula_str:
        return None
    try:
        # Substituições para o SymPy
        formula_str = formula_str.replace('^', '&').replace('v', '|').replace('->', '>>').replace('<->', '=').replace('~', '~')
        return parse_expr(formula_str, evaluate=False)
    except Exception:
        return None

def converter_notacao_padrao(formula):
    """
    Converte o objeto SymPy para uma string em notação lógica padrão (¬, ∧, ∨, →)
    usando recursão para garantir parentização correta.
    """
    if formula == sympy.false:
        return '⊥'
    if formula == sympy.true:
        return '⊤'

    if isinstance(formula, Symbol):
        return str(formula)

    if isinstance(formula, Not):
        arg_str = converter_notacao_padrao(formula.args[0])
        # Adiciona parênteses apenas se o argumento for uma expressão composta
        if isinstance(formula.args[0], (Implies, And, Or, Not)):
            return f"¬({arg_str})"
        return f"¬{arg_str}"

    if isinstance(formula, And):
        op = '∧'
    elif isinstance(formula, Or):
        op = '∨'
    elif isinstance(formula, Implies):
        op = '→'
    else:
        return str(formula)

    # Converte argumentos recursivamente e combina com parênteses obrigatórios
    arg1_str = converter_notacao_padrao(formula.args[0])
    arg2_str = converter_notacao_padrao(formula.args[1])

    return f"({arg1_str} {op} {arg2_str})"


def is_entailed_by(premisses, conclusion):
    """Oráculo SymPy: Verifica se a conclusão é uma consequência lógica das premissas."""
    if not premisses:
        return sympy.satisfiable(Not(conclusion)) is False
    return sympy.satisfiable(And(*premisses, Not(conclusion))) is False

def get_accessible_formulas_and_lines(data, current_level):
    """Retorna as fórmulas acessíveis em um determinado nível de prova (escopo de suposição)."""
    accessible = {}
    for level, line, f_obj, f_str, just in data:
        if level <= current_level:
            accessible[f_obj] = line
    return accessible


# 3. MOTOR DE DEDUÇÃO NATURAL (DEDUCTION ENGINE)

def _search_and_apply_rules(data_snapshot, target_conclusion_obj, max_steps):
    """Algoritmo recursivo central para busca de provas em Dedução Natural (DN)."""
    if max_steps <= 0:
        return data_snapshot, len(data_snapshot), False

    data = list(data_snapshot)
    current_level = data[-1][0] if data else 1
    line_num_base = max(step[1] for step in data) + 1 if data else 1

    accessible_formulas = get_accessible_formulas_and_lines(data, current_level)
    accessible_set = set(accessible_formulas.keys())
    existing_formulas = accessible_set.copy()

    current_accessible_premisses_str = tuple(sorted([converter_notacao_padrao(f) for f in accessible_set]))
    cache_key = (target_conclusion_obj, current_accessible_premisses_str)

    if cache_key in TARGET_CACHE:
        cached_data, cached_line_num = TARGET_CACHE[cache_key]
        offset = line_num_base - cached_data[0][1] if cached_data else 0
        for level, old_line, f_obj, f_str, just in cached_data:
            new_line = old_line + offset
            data.append([level, new_line, f_obj, f_str, just])
            line_num_base = new_line + 1
        return data, line_num_base, True

    if target_conclusion_obj in accessible_set:
        return data, line_num_base, True

    while True:
        progress_made = False

        if isinstance(target_conclusion_obj, And):
            A, B = target_conclusion_obj.args[0], target_conclusion_obj.args[1]
            if A in accessible_set and B in accessible_set:
                ref_a = accessible_formulas[A]
                ref_b = accessible_formulas[B]
                just_str = f"∧I ({ref_a}, {ref_b})"
                target_str = converter_notacao_padrao(target_conclusion_obj)
                data.append([current_level, line_num_base, target_conclusion_obj, target_str, just_str])
                line_num_base += 1
                return data, line_num_base, True

        new_step_added = False

        if sympy.false not in accessible_set:
            for f in accessible_set:
                if Not(f) in accessible_set:
                    ref_f = accessible_formulas[f]
                    ref_not_f = accessible_formulas[Not(f)]
                    new_formula_obj = sympy.false
                    if new_formula_obj not in existing_formulas:
                        new_line = [current_level, line_num_base, new_formula_obj, '⊥', f"⊥I ({ref_f}, {ref_not_f})"]
                        data.append(new_line)
                        accessible_formulas[new_formula_obj] = line_num_base
                        accessible_set.add(new_formula_obj)
                        existing_formulas.add(new_formula_obj)
                        line_num_base += 1
                        progress_made = True
                        new_step_added = True
                        break
            if new_step_added: continue

        if sympy.false in accessible_set and target_conclusion_obj not in accessible_set:
            ref_bot = accessible_formulas[sympy.false]
            new_formula_obj = target_conclusion_obj
            if new_formula_obj not in existing_formulas:
                target_str = converter_notacao_padrao(new_formula_obj)
                data.append([current_level, line_num_base, new_formula_obj, target_str, f"⊥E ({ref_bot})"])
                line_num_base += 1
                return data, line_num_base, True

        added_something_direct = False
        for f_current, ref_current in list(accessible_formulas.items()):

            if isinstance(f_current, Implies):
                antecedent, consequent = f_current.args[0], f_current.args[1]
                if antecedent in accessible_set and consequent not in existing_formulas:
                    ref_ant = accessible_formulas[antecedent]
                    new_formula_obj = consequent
                    consequent_str = converter_notacao_padrao(new_formula_obj)

                    data.append([current_level, line_num_base, new_formula_obj, consequent_str, f"→E ({ref_current}, {ref_ant})"])
                    accessible_formulas[new_formula_obj] = line_num_base
                    accessible_set.add(new_formula_obj)
                    existing_formulas.add(new_formula_obj)
                    line_num_base += 1
                    added_something_direct = True
                    break

            elif isinstance(f_current, And):
                A, B = f_current.args[0], f_current.args[1]
                added_parts = False
                for part in [A, B]:
                    if part not in existing_formulas:
                        part_str = converter_notacao_padrao(part)
                        data.append([current_level, line_num_base, part, part_str, f"∧E ({ref_current})"])
                        accessible_formulas[part] = line_num_base
                        accessible_set.add(part)
                        existing_formulas.add(part)
                        line_num_base += 1
                        added_something_direct = True
                        added_parts = True
                if added_parts:
                    break

            elif isinstance(f_current, Not) and isinstance(f_current.args[0], Not):
                new_formula_obj = f_current.args[0].args[0]
                if new_formula_obj not in existing_formulas:
                    A_str = converter_notacao_padrao(new_formula_obj)
                    data.append([current_level, line_num_base, new_formula_obj, A_str, f"¬¬E ({ref_current})"])
                    accessible_formulas[new_formula_obj] = line_num_base
                    accessible_set.add(new_formula_obj)
                    existing_formulas.add(new_formula_obj)
                    line_num_base += 1
                    added_something_direct = True
                    break

        if added_something_direct:
            progress_made = True
            continue

        break

    if target_conclusion_obj in accessible_set:
        return data, line_num_base, True


    disjunction_candidates = [(f_dis, ref_dis) for f_dis, ref_dis in accessible_formulas.items() if isinstance(f_dis, Or)]

    if disjunction_candidates:
        f_dis, ref_dis = disjunction_candidates[0]
        A, B = f_dis.args[0], f_dis.args[1]

        sup_A_start = line_num_base
        sup_A_str = converter_notacao_padrao(A)

        data_sup_A = data + [[current_level + 1, sup_A_start, A, sup_A_str, "SUP ∨E (A)"]]
        data_sub_A, line_num_sub_A, found_A_to_C = _search_and_apply_rules(data_sup_A, target_conclusion_obj, max_steps - 1)

        if found_A_to_C:
            sup_A_end = line_num_sub_A - 1
            steps_A = [step for step in data_sub_A if sup_A_start <= step[1] <= sup_A_end]

            sup_B_start = line_num_sub_A
            sup_B_str = converter_notacao_padrao(B)

            data_base_B = [step for step in data if step[0] <= current_level]
            data_base_B.extend(steps_A)
            data_sup_B = data_base_B + [[current_level + 1, sup_B_start, B, sup_B_str, "SUP ∨E (B)"]]

            data_sub_B, line_num_sub_B, found_B_to_C = _search_and_apply_rules(data_sup_B, target_conclusion_obj, max_steps - 1)

            if found_B_to_C:
                data = data_sub_B
                line_num_base = line_num_sub_B
                sup_B_end = line_num_base - 1

                conc_str = converter_notacao_padrao(target_conclusion_obj)
                just_str = f"∨E ({ref_dis}, {sup_A_start}-{sup_A_end}, {sup_B_start}-{sup_B_end})"

                data.append([current_level, line_num_base, target_conclusion_obj, conc_str, just_str])
                line_num_base += 1

                cache_steps_start = data_snapshot[-1][1] + 1 if data_snapshot else 1
                cache_steps = [step for step in data if step[1] >= cache_steps_start]
                TARGET_CACHE[cache_key] = (cache_steps, line_num_base)

                return data, line_num_base, True


    if isinstance(target_conclusion_obj, Implies):
        antecedent_obj = target_conclusion_obj.args[0]
        consequente_obj = target_conclusion_obj.args[1]
        antecedent_str = converter_notacao_padrao(antecedent_obj)

        sup_start = line_num_base
        data_sup = data + [[current_level + 1, sup_start, antecedent_obj, antecedent_str, "SUP →I"]]

        data_sub, line_num_sub, found_consequente = _search_and_apply_rules(data_sup, consequente_obj, max_steps - 1)

        if found_consequente:
            data = data_sub
            line_num_base = line_num_sub
            sup_end = line_num_base - 1

            conc_str = converter_notacao_padrao(target_conclusion_obj)
            just_str = f"→I ({sup_start}-{sup_end})"

            data.append([current_level, line_num_base, target_conclusion_obj, conc_str, just_str])
            line_num_base += 1

            cache_steps = [step for step in data if sup_start <= step[1] <= sup_end]
            TARGET_CACHE[cache_key] = (cache_steps, sup_end + 1)

            return data, line_num_base, True

    if target_conclusion_obj not in accessible_set:
        target_A = target_conclusion_obj

        if isinstance(target_A, Not):
            assumption_formula = target_A.args[0]
            just_tag, rule_tag = "SUP ¬I", "¬I"
        else:
            assumption_formula = Not(target_A)
            just_tag, rule_tag = "SUP RAA", "RAA"

        assumption_str = converter_notacao_padrao(assumption_formula)
        sup_start_neg = line_num_base

        data_neg = data + [[current_level + 1, sup_start_neg, assumption_formula, assumption_str, just_tag]]

        data_sub_neg, line_num_sub_neg, found_bot = _search_and_apply_rules(data_neg, sympy.false, max_steps - 1)

        if found_bot:
            data = data_sub_neg
            line_num_base = line_num_sub_neg
            sup_end_neg = line_num_base - 1
            conc_str = converter_notacao_padrao(target_A)
            just_str = f"{rule_tag} ({sup_start_neg}-{sup_end_neg})"

            data.append([current_level, line_num_base, target_A, conc_str, just_str])
            line_num_base += 1

            cache_steps = [step for step in data if sup_start_neg <= step[1] <= sup_end_neg]
            TARGET_CACHE[cache_key] = (cache_steps, sup_end_neg + 1)

            return data, line_num_base, True

    return data, line_num_base, target_conclusion_obj in set(accessible_formulas.keys())


# 4. MOTOR DE TABLEAUX

def _expand_tableau_demonstration(formula_sympy, current_level, ref_line, line_num, tableaux_steps):
    """
    Aplica a regra de Tableaux recursivamente e armazena os passos.
    """

    # Lógica de Fechamento (Procura por A e ¬A no caminho)
    is_closed = False
    current_accessible_formulas = {step[2] for step in tableaux_steps if step[0] <= current_level and step[2] is not None}
    ref_line_closed = None

    if formula_sympy.is_Atom and Not(formula_sympy) in current_accessible_formulas:
        is_closed = True
        ref_line_closed = [step[1] for step in tableaux_steps if step[2] == Not(formula_sympy) and step[0] <= current_level][0]
    elif isinstance(formula_sympy, Not) and formula_sympy.args[0].is_Atom and formula_sympy.args[0] in current_accessible_formulas:
        is_closed = True
        ref_line_closed = [step[1] for step in tableaux_steps if step[2] == formula_sympy.args[0] and step[0] <= current_level][0]

    if is_closed:
        # Passo de fechamento
        tableaux_steps.append(
            [current_level, line_num, None, "X", f"Fechamento (Linhas {ref_line}, {ref_line_closed})", False]
        )
        return tableaux_steps, line_num + 1

    # --- Regras Alfa (Não Bifurcam) ---
    rule = None
    components = []

    if isinstance(formula_sympy, Not) and isinstance(formula_sympy.args[0], Implies):
        rule = "α (¬→)"
        A = formula_sympy.args[0].args[0]
        B = formula_sympy.args[0].args[1]
        components = [A, Not(B)]

    elif isinstance(formula_sympy, And):
        rule = "α (∧)"
        A, B = formula_sympy.args[0], formula_sympy.args[1]
        components = [A, B]

    elif isinstance(formula_sympy, Not) and isinstance(formula_sympy.args[0], Or):
        rule = "α (¬∨)"
        A, B = formula_sympy.args[0].args[0], formula_sympy.args[0].args[1]
        components = [Not(A), Not(B)]

    elif isinstance(formula_sympy, Not) and isinstance(formula_sympy.args[0], Not):
        rule = "α (¬¬)"
        A = formula_sympy.args[0].args[0]
        components = [A]

    if rule and not components:
        pass
    elif rule:
        # Aplica os componentes Alfa em sequência
        for comp in components:
            tableaux_steps.append([current_level, line_num, comp, converter_notacao_padrao(comp), f"{rule} ({ref_line})", True])
            line_num += 1
            # Continua a expansão no mesmo ramo (mesmo current_level)
            tableaux_steps, line_num = _expand_tableau_demonstration(comp, current_level, line_num - 1, line_num, tableaux_steps)
        return tableaux_steps, line_num

    # --- Regras Beta (Bifurcam) ---
    beta_rule = None
    beta_components = []

    if isinstance(formula_sympy, Implies):
        beta_rule = "β (→)"
        A, B = formula_sympy.args[0], formula_sympy.args[1]
        beta_components = [Not(A), B]

    elif isinstance(formula_sympy, Or):
        beta_rule = "β (∨)"
        A, B = formula_sympy.args[0], formula_sympy.args[1]
        beta_components = [A, B]

    elif isinstance(formula_sympy, Not) and isinstance(formula_sympy.args[0], And):
        beta_rule = "β (¬∧)"
        A, B = formula_sympy.args[0].args[0], formula_sympy.args[0].args[1]
        beta_components = [Not(A), Not(B)]

    if beta_rule:
        comp_A, comp_B = beta_components[0], beta_components[1]

        # 1. Passo de Comentário da Bifurcação (novo passo, nova linha)
        tableaux_steps.append(
            [current_level, line_num, None, "Bifurcação", f"Regra {beta_rule} aplicada à linha {ref_line}", True]
        )
        line_num += 1

        # 2. Passo de Linha Gráfica (novo passo, nova linha)
        tableaux_steps.append(
            [current_level, line_num, None, "Linha Gráfica", f"Bifurcação Nível {current_level} para Nível {current_level+1}", True]
        )
        line_num += 1


        # O histórico comum é apenas o que estava acessível ANTES da bifurcação
        base_steps = [step for step in tableaux_steps if step[0] <= current_level]

        # --- RAMO A (Esquerdo) ---
        line_A_start = line_num
        steps_A = list(base_steps) + [[current_level + 1, line_A_start, comp_A, converter_notacao_padrao(comp_A), f"Ramo Esquerdo {beta_rule} ({ref_line})", True]]
        line_A_end = line_A_start + 1
        # Expande recursivamente o Ramo A (novo nível: current_level + 1)
        steps_A, line_A_end = _expand_tableau_demonstration(comp_A, current_level + 1, line_A_start, line_A_end, steps_A)

        # --- RAMO B (Direito) ---
        line_B_start = line_A_end

        # Base de Steps para Ramo B: Passos de Nível<=current_level + Passos do Ramo A
        steps_B_base = [step for step in steps_A if step[0] <= current_level]
        # Aqui, Ramo B herda o histórico do Ramo A. Se Ramo A fechou, Ramo B continua do último passo comum.

        steps_B = list(steps_B_base) + [[current_level + 1, line_B_start, comp_B, converter_notacao_padrao(comp_B), f"Ramo Direito {beta_rule} ({ref_line})", True]]
        line_B_end = line_B_start + 1
        # Expande recursivamente o Ramo B (novo nível: current_level + 1)
        steps_B, line_B_end = _expand_tableau_demonstration(comp_B, current_level + 1, line_B_start, line_B_end, steps_B)

        # Retorna o resultado da expansão do Ramo B (que contém A e o caminho comum)
        return steps_B, line_B_end

    return tableaux_steps, line_num


def analisar_tableaux(formula_refutacao_sympy):
    """Orquestra a expansão do Tableaux Semântico."""

    is_satisfiable = sympy.satisfiable(formula_refutacao_sympy)
    is_valid_from_tableaux = not is_satisfiable

    tableaux_steps = []
    line_num = 1
    current_level = 1

    formula_str = converter_notacao_padrao(formula_refutacao_sympy)

    # 1. Fórmula de Refutação
    tableaux_steps.append(
        [current_level, line_num, formula_refutacao_sympy, formula_str, "Fórmula de Refutação (P)", True]
    )

    line_num += 1

    # 2. Expansão recursiva
    tableaux_steps, line_num = _expand_tableau_demonstration(formula_refutacao_sympy, current_level, 1, line_num, tableaux_steps)

    # 3. Tratamento de ramos abertos
    has_closed_branch = any(step[3] == "X" for step in tableaux_steps)

    if not is_valid_from_tableaux and not has_closed_branch:
         tableaux_steps.append(
            [current_level, line_num, None, "O", "Ramo Aberto (Conclusão)", True]
        )

    if is_valid_from_tableaux and not has_closed_branch:
        # Se SymPy diz que é válido (fecha), mas o Tableaux não conseguiu fechar (profundidade máxima atingida?), forçamos o fechamento.
        tableaux_steps.append(
            [current_level, line_num, None, "X", "Fechamento (Oráculo SymPy)", False]
        )

    return tableaux_steps, is_valid_from_tableaux


# 5. APRESENTAÇÃO (RENDERER)

def imprimir_prova_dn(proof_steps_data, premissas_padrao, conclusao_padrao, is_valid):
    """
    Renderiza a prova de Dedução Natural usando HTML/CSS.
    Corrigido o símbolo $\vdash$ para ⊢.
    """

    argumento_premissas_str = ', '.join(premissas_padrao)
    argumento_completo_str = f"{argumento_premissas_str} ⊢ {conclusao_padrao}"

    html_output = f"""
    <h2>PROVA: DEDUÇÃO NATURAL (DN)</h2>
    <p>Argumento: <b>{argumento_completo_str}</b></p>
    <p>Validade Lógica (SymPy): <b>{'Válido' if is_valid else 'Inválido'}</b></p>
    """

    df_data = []

    for level, line, f_obj, f_str, just in proof_steps_data:

        # 1. Linhas Verticais de Escopo (Indentações)
        indent_html = ""
        for i in range(1, level):
             # Cor da linha de escopo
            indent_html += f'<div style="border-left: 2px solid #ccc; width: 15px; height: 100%; display: inline-block; margin-right: 5px;"></div>'

        is_assumption = "SUP" in just and level > 1

        # 2. Fórmula formatada
        formula_content = f'{indent_html}<span style="font-family: monospace; font-size: 1.1em; white-space: pre;">{f_str}</span>'

        # 3. Justificativa formatada (Cor branca para alto contraste em fundo escuro)
        just_color = "white"

        just_content = f'<span style="font-family: monospace; color: {just_color}; white-space: nowrap;">{just}</span>'

        df_data.append([
            line,
            formula_content,
            just_content
        ])

        # Adiciona a linha tracejada no início da suposição, se necessário
        if is_assumption:
             df_data[-1][1] = f'{indent_html}<span style="border-top: 1px dashed #777; width: 100%; display: block; margin-bottom: -10px;"></span>' + df_data[-1][1]

        # Adiciona a linha horizontal para a conclusão de subprovas
        if "I (" in just or "RAA (" in just:
            df_data[-1][1] = f'{indent_html}<span style="border-top: 2px solid #000; width: 100%; display: block; margin-bottom: -10px;"></span>' + df_data[-1][1]


    df = pd.DataFrame(df_data, columns=['Linha', 'Fórmula', 'Justificativa'])

    # 4. Renderiza o DataFrame como HTML

    html_table = df.to_html(
        index=False,
        escape=False,
        header=True,
        classes=['proof-table'],
    )

    # CSS para estilizar a tabela
    css_style = """
    <style>
    .proof-table {
        border-collapse: collapse;
        width: auto;
        font-size: 14px;
        margin-top: 15px;
    }
    .proof-table th {
        border-bottom: 2px solid #000;
        padding: 5px 10px;
        text-align: left;
        font-weight: bold;
    }
    .proof-table td {
        padding: 5px 10px;
        vertical-align: top;
    }
    .proof-table tr:not(:last-child) td {
        border-bottom: none;
    }
    </style>
    """

    display(HTML(css_style + html_output + html_table))


def visualizar_tableaux_sistematico(tableaux_steps_data, formula_refutacao, resultado_tableaux):
    """
    [CORRIGIDO] Renderiza a prova de Tableaux Semânticos.
    - Corrigido o problema de linhas pulando.
    - Aumentado o espaçamento (padding) para evitar fórmulas cortadas.
    """

    html_output = f"""
    <h2>PROVA: TABLEAUX SEMÂNTICOS</h2>
    <p>Fórmula de Refutação (¬Argumento): <b>{formula_refutacao}</b></p>
    <p>Resultado Tableaux: <b>{resultado_tableaux}</b></p>
    <h3>Visualização da Árvore Tableaux</h3>
    """

    df_data = []

    for i, step in enumerate(tableaux_steps_data):

        current_level = step[0]
        line_num = step[1]
        formula_str = step[3]
        justificativa = step[4]

        # --- Lógica de Linhas de Conexão e Indentação ---
        indent_html = ""
        line_color = "#333" # Cor das linhas verticais (escuras)

        is_graphic_line = (formula_str == "Linha Gráfica")
        is_bifurcation_comment = (formula_str == "Bifurcação")

        if current_level > 1 and not is_graphic_line:
            # Desenha as linhas verticais de escopo para o passo atual
            for level_i in range(1, current_level):
                indent_html += f'<span style="display: inline-block; width: 25px; text-align: center; color: {line_color};">|</span>'


        # 1. TRATAMENTO DE BIFURCAÇÃO (Linha de Comentário)
        if is_bifurcation_comment:
            just_bif = justificativa.split(" aplicada à linha ")[0]
            # Usa o espaço de nível anterior (current_level)
            df_data.append([
                line_num,
                f'{indent_html}<span style="display: inline-block; width: 50px; text-align: left; font-style: italic; color: white;">{just_bif}</span>',
                f'<span style="font-family: monospace; color: white;">Ref: {justificativa.split("linha ")[1]}</span>'
            ])
            continue

        # 2. TRATAMENTO DA LINHA GRÁFICA ( / \ )
        if is_graphic_line:
            # A linha gráfica usa a indentação do nível anterior (current_level - 1)
            indent_bif_line = ""
            for level_i in range(1, current_level):
                 indent_bif_line += f'<span style="display: inline-block; width: 25px; text-align: center; color: {line_color};">|</span>'

            # Desenha o separador / \ forte (preto)
            df_data.append([
                 "", # Sem número de linha para a linha gráfica
                 f'{indent_bif_line}<span style="display: inline-block; width: 25px; text-align: center; color: #000; font-weight: bold;">/ \\</span>',
                 ""
            ])
            continue


        # 3. TRATAMENTO DE RAMOS (Fórmulas e Resultados)

        # Adiciona o símbolo de ramo (Ramo Esquerdo/Direito)
        # Símbolos de conexão fortes (Preto)
        branch_symbol = ""

        # O Ramo Esquerdo/Direito é sempre no nível "current_level"
        if "Ramo Esquerdo" in justificativa:
            branch_symbol = f'<span style="display: inline-block; width: 25px; text-align: center; color: #000; font-weight: bold;">├──</span>'
            indent_html = indent_html[:-25] + branch_symbol

        elif "Ramo Direito" in justificativa:
            branch_symbol = f'<span style="display: inline-block; width: 25px; text-align: center; color: #000; font-weight: bold;">└──</span>'
            indent_html = indent_html[:-25] + branch_symbol


        # 4. Conteúdo da Fórmula e Justificativa
        formula_display = formula_str
        just_display = justificativa

        # 5. Fechamento/Abertura (Cores de status)
        formula_style = 'font-weight: normal;'
        just_style = 'font-weight: normal; color: white;'

        if formula_str == "X":
            formula_display = "❌ FECHADO"
            formula_style = 'font-weight: bold; color: red;'
            just_style = 'font-weight: bold; color: red;'
        elif formula_str == "O":
            formula_display = "🟢 ABERTO"
            formula_style = 'font-weight: bold; color: green;'
            just_style = 'font-weight: bold; color: green;'


        formula_content = f'{indent_html}<span style="font-family: monospace; font-size: 1.1em; white-space: pre; {formula_style}">{formula_display}</span>'
        just_content = f'<span style="font-family: monospace; white-space: nowrap; {just_style}">{just_display}</span>'

        df_data.append([
            line_num,
            formula_content,
            just_content
        ])


    df = pd.DataFrame(df_data, columns=['Linha', 'Fórmula', 'Justificativa'])

    # 6. Renderiza a tabela Tableaux
    html_table = df.to_html(
        index=False,
        escape=False,
        header=True,
        classes=['tableau-table'],
    )

    # CSS para estilizar a tabela
    css_style = """
    <style>
    .tableau-table {
        border-collapse: collapse;
        width: auto;
        font-size: 14px;
        margin-top: 15px;
    }
    .tableau-table th {
        border-bottom: 2px solid #000;
        padding: 5px 10px;
        text-align: left;
        font-weight: bold;
    }
    /* CORREÇÃO AQUI: Aumento do padding vertical e horizontal para evitar cortes */
    .tableau-table td {
        padding: 5px 10px;
        vertical-align: top;
        border-bottom: none;
    }
    </style>
    """

    conclusion_msg = "Todos os ramos Fechados (❌) → Argumento Válido." if resultado_tableaux == "Fechou (Válido)" else "Pelo menos um ramo Aberto (🟢) → Argumento Inválido."

    display(HTML(css_style + html_output + f"<p>Conclusão Tableaux: {conclusion_msg}</p>" + html_table))


# 6. CONTROLADOR E INTERFACE (CONTROLLER)

def prova_por_dn(premissas_obj, conclusao_obj, premissas_padrao, conclusao_padrao, is_valid_dn):
    initial_data = []
    line_num = 1
    for p_obj, p_str in zip(premissas_obj, premissas_padrao):
        initial_data.append([1, line_num, p_obj, p_str, "PREMISSA"])
        line_num += 1

    proof_steps_data, _, _ = _search_and_apply_rules(initial_data, conclusao_obj, max_steps=350)
    imprimir_prova_dn(proof_steps_data, premissas_padrao, conclusao_padrao, is_valid_dn)


def prova_por_tableaux(premissas_obj, conclusao_obj):
    if not premissas_obj:
        antecedente_conjunto = sympy.true
    elif len(premissas_obj) == 1:
        antecedente_conjunto = premissas_obj[0]
    else:
        antecedente_conjunto = And(*premissas_obj)

    argumento_implicacao = Implies(antecedente_conjunto, conclusao_obj)
    refutacao_sympy = Not(argumento_implicacao)
    formula_inicial_notacao_padrao = converter_notacao_padrao(refutacao_sympy)

    tableaux_data, is_valid_from_tableaux = analisar_tableaux(refutacao_sympy)

    resultado_string = "Fechou (Válido)" if is_valid_from_tableaux else "Ramo Aberto (Inválido)"

    visualizar_tableaux_sistematico(
        tableaux_data,
        formula_inicial_notacao_padrao,
        resultado_string
    )


def prova_logica_generica(premissas_str, conclusao_str):
    global TARGET_CACHE
    TARGET_CACHE = {}

    premissas_obj = [parse_formula_logica(p) for p in premissas_str if p.strip()]
    conclusao_obj = parse_formula_logica(conclusao_str)

    if any(p is None for p in premissas_obj) or conclusao_obj is None:
        print("\nERRO: Uma ou mais fórmulas são sintaticamente inválidas.")
        return

    is_valid_dn = is_entailed_by(premissas_obj, conclusao_obj)

    premissas_padrao = [converter_notacao_padrao(p) for p in premissas_obj]
    conclusao_padrao = converter_notacao_padrao(conclusao_obj)

    prova_por_dn(premissas_obj, conclusao_obj, premissas_padrao, conclusao_padrao, is_valid_dn)

    prova_por_tableaux(premissas_obj, conclusao_obj)


# MODO 1: Testes Pré-Configurados

def interface_testes():
    """Interface para rodar as expressões pré-configuradas para teste."""

    testes_argumentos = [
        (["A->B", "A"], "B", "Teste 1: Modus Ponens (Válido)"),
        (["~A v B", "A"], "B", "Teste 2: Silogismo Disjuntivo (Válido)"),
        (["A->~A"], "~A", "Teste 3: Redução ao Absurdo (RAA) (Válido)"),
        (["A v B", "A->C", "B->C"], "C", "Teste 4: Eliminação da Disjunção (∨E)"),
        (["~~A"], "A", "Teste 5: Eliminação da Dupla Negação (~~E) (Válido)"),
        (["A->B", "B"], "A", "Teste 6: Falácia da Afirmação do Consequente (Inválido)"),
        (["A"], "B->A", "Teste 7: Introdução da Implicação (→I) (Válido)")
    ]

    print("--- INÍCIO DOS TESTES EMBUTIDOS ---")

    for i, (premissas, conclusao, nome_teste) in enumerate(testes_argumentos, 1):
        print(f"\n\n\nINICIANDO {nome_teste}")

        prova_logica_generica(premissas, conclusao)

        print(f"\nFIM DO {nome_teste}")

    print("\n--- FIM DOS TESTES EMBUTIDOS ---")


# MODO 2: Entrada do Usuário

def resolver_argumento_usuario():
    """
    Função para coletar a entrada do usuário e resolver o argumento
    usando Dedução Natural e Tableaux Semântico.
    """
    print("\n--- SOLUCIONADOR LÓGICO MANUAL ---")
    print("Use a seguinte notação:")
    print("  E/AND: ^ ou &")
    print("  OU/OR: v ou |")
    print("  IMPLICAÇÃO: ->")
    print("  NEGAÇÃO/NOT: ~")
    print("  Exemplo de premissas: A->B, A")

    try:
        premissas_input = input("\nDigite as PREMISSAS (separadas por vírgula): ")
        conclusao_input = input("Digite a CONCLUSÃO: ")
    except EOFError:
        print("\nEntrada interrompida. Finalizando.")
        return
    except Exception as e:
        print(f"\nOcorreu um erro na entrada: {e}")
        return


    premissas_list = [p.strip() for p in premissas_input.split(',') if p.strip()]

    if not conclusao_input:
        print("\nERRO: A conclusão não pode ser vazia.")
        return

    print("\n\nRESOLVENDO ARGUMENTO...")
    prova_logica_generica(premissas_list, conclusao_input)


# MENU DE INÍCIO

def iniciar_solucionador_logico():
    """Menu para escolher o modo de execução."""
    print("=========================================")
    print("    SISTEMA DE PROVAS LÓGICAS (DN/TABLEAUX)")
    print("=========================================")
    print("Escolha o modo de execução:")
    print("1. Rodar Testes Pré-Configurados")
    print("2. Inserir Argumento Manualmente")

    try:
        escolha = input("Digite 1 ou 2: ")

        if escolha == '1':
            interface_testes()
        elif escolha == '2':
            resolver_argumento_usuario()
        else:
            print("Opção inválida. Por favor, digite 1 ou 2.")
    except Exception:
        print("\nEntrada inválida. Finalizando.")

Para iniciar o sistema, chame: iniciar_solucionador_logico()

1.   interface_testes(): Executa argumentos pré-definidos (Modus Ponens, RAA, etc.)
2.   resolver_argumento_usuario(): Solicita as premissas e a conclusão ao usuário.

In [38]:
iniciar_solucionador_logico()

    SISTEMA DE PROVAS LÓGICAS (DN/TABLEAUX)
Escolha o modo de execução:
1. Rodar Testes Pré-Configurados
2. Inserir Argumento Manualmente
Digite 1 ou 2: 2

--- SOLUCIONADOR LÓGICO MANUAL ---
Use a seguinte notação:
  E/AND: ^ ou &
  OU/OR: v ou |
  IMPLICAÇÃO: ->
  NEGAÇÃO/NOT: ~
  Exemplo de premissas: A->B, A

Digite as PREMISSAS (separadas por vírgula): A
Digite a CONCLUSÃO: B->A


RESOLVENDO ARGUMENTO...


Linha,Fórmula,Justificativa
1,A,PREMISSA
2,B,SUP →I
3,(B → A),→I (2-2)


Linha,Fórmula,Justificativa
1,¬((A → (B → A))),Fórmula de Refutação (P)
2,A,α (¬→) (1)
3,¬((B → A)),α (¬→) (1)
4,B,α (¬→) (3)
5,¬A,α (¬→) (3)
6,❌ FECHADO,"Fechamento (Linhas 5, 2)"
